In [1]:
from cgra import *
from kernels import *

In [2]:
kernel_name = "benchmarks/compigra_blas_paper/blas/mmul"
version = "_4_IJK24"

In [3]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------            
    first_addr_A = first_addr
    first_addr_B = first_addr_A + rowsA*colsA*4
    first_addr_C = first_addr_B + colsA*colsB*4

    config_vals_col0 = [first_addr_A]
    config_vals_col1 = []
    config_vals_col2 = []
    config_vals_col3 = [first_addr_C, first_addr_B]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [6]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [7]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [8]:
def mmul_cpu(A_data, B_data, C_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum 
    return expected_res

In [9]:
# Test dimensions (4xXx4)
rowsA = 24
colsA = 24
colsB = 24
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
C_data = [x + 200 for x in range(0, rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
C_data_cpy = C_data.copy()

#print("A")
#printAsMatrix(A_data, rowsA, colsA)
#print("B")
#printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB)

In [10]:
runKernel(load_addrs, max_it=200000)

Instr =  0 ( 0 )
[20000,   15,    0,   15]    [LWD ROUT  4, SADD R0  15 ZERO, NOP , SADD R0  15 ZERO]    
[  15,    0,   15, 24608]    [SADD R0  15 ZERO, NOP , SADD ROUT  15 ZERO, LWD R0  4]    
[  24,   24,    0,   15]    [SADD ROUT  ZERO 24, SADD ROUT  ZERO 24, NOP , SADD R0  15 ZERO]    
[  15,   24,    0, 22304]    [SADD ROUT  15 ZERO, SADD R0  ZERO 24, NOP , LWD ROUT  4]    
-------
Instr =  1 ( 1 )
[20000, 61440, 61440,   15]    [NOP , SLT R0  R0 12, SLT R0  RCB 12, NOP ]    
[61440,    0,   15, 24608]    [SLT ROUT  R0 12, NOP , NOP , NOP ]    
[  24,   24,    0, 61440]    [NOP , NOP , NOP , SLT ROUT  R0 12]    
[  15, 61440,    0, 22304]    [NOP , SLT ROUT  RCL 12, NOP , NOP ]    
-------
Instr =  2 ( 2 )
[20000, 65040, 65024, 61440]    [NOP , SADD ROUT  3600 R0, SADD ROUT  3584 R0, SLT R0  R0 12]    
[61440, 65036,   15, 24608]    [NOP , SADD ROUT  3596 RCL, NOP , NOP ]    
[  24,   24, 65028, 61440]    [NOP , NOP , SADD ROUT  3588 RCR, NOP ]    
[  15, 61440, 65044, 22304]    

In [11]:
# Get result from CGRA
first_addr_C = first_addr + rowsA*colsA*4 + colsA*colsB*4
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)
# Process estra rows/cols
if rowsA%CGRA_N_COLS != 0:
    for rA in range(rowsA - rowsA%CGRA_N_COLS, rowsA):
        for cB in range(colsB):
            sum = 0
            for k in range(colsA):
                sum += A_data[rA*colsA+k]*B_data[k*colsB+cB]
            result[rA*colsB+cB] = sum 
if colsB%CGRA_N_COLS != 0:
    for cB in range(colsB - colsB%CGRA_N_COLS, colsB):
        for rA in range(rowsA - rowsA%CGRA_N_COLS):
            sum = 0
            for k in range(colsA):
                sum += A_data[rA*colsA+k]*B_data[k*colsB+cB]
            result[rA*colsB+cB] = sum

# Get cpu output
expected_res = mmul_cpu(A_data_cpy, B_data_cpy, C_data_cpy, rowsA, colsA, colsB)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("CGRA: ")
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")



OK
